#1. 프로젝트 폴더 생성

In [ ]:
!mkdir -p ai_speech_pipeline/config
!mkdir -p ai_speech_pipeline/services
!mkdir -p ai_speech_pipeline/api
!mkdir -p ai_speech_pipeline/data/input
!mkdir -p ai_speech_pipeline/data/output

%cd ai_speech_pipeline

#2. requirements.txt 작성
설치 패키지 관리



In [ ]:
%%writefile requirements.txt
fastapi
uvicorn
nest-asyncio
pyngrok
openai
pydub
jiwer
requests
openai-whisper

#3. 패키지 설치

In [ ]:
!pip install -q -r requirements.txt
!apt-get update -qq
!apt-get install -y ffmpeg

#4. config/settings.py

In [ ]:
%%writefile config/settings.py
from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
COLAB_API_TOKEN = userdata.get("COLAB_API_TOKEN")

#6. services/scenario_service.py

In [ ]:
%%writefile services/scenario_service.py
import json
from openai import OpenAI
from config.settings import OPENAI_API_KEY


client = OpenAI(api_key=OPENAI_API_KEY)


def generate_scenario_levels(scenario_context: str, goal: str) -> dict:
    system_prompt = """
너는 성인 의사소통 훈련용 시나리오 설계 전문가임.

사용자가 입력한 시나리오 상황과 목적을 바탕으로,
의사소통 연습이 가능한 3단계 난이도(Level 1, Level 2, Level 3)를 설계해야 함.

규칙:
1. 총 3개 Level을 만들어야 함.
2. 각 Level마다 Step 1, Step 2, Step 3을 만들어야 함.
3. 총 9개의 step이 생성되어야 함.
4. 난이도는 Level 1 -> Level 2 -> Level 3 순으로 점진적으로 올라가야 함.
5. 각 Step은 실제 대화 흐름처럼 자연스럽게 이어져야 함.
6. 각 Step에는 아래 정보가 포함되어야 함.
   - step
   - assistantMessage
   - userIntent
7. 결과는 반드시 JSON만 반환해야 함.
8. 한국어로 작성해야 함.
"""

    user_prompt = f"""
시나리오 상황: {scenario_context}
사용자 목적: {goal}

아래 JSON 형식으로만 반환:
{{
  "scenarioContext": "{scenario_context}",
  "goal": "{goal}",
  "levels": [
    {{
      "level": 1,
      "levelTitle": "...",
      "levelDescription": "...",
      "steps": [
        {{"step": 1, "assistantMessage": "...", "userIntent": "..."}},
        {{"step": 2, "assistantMessage": "...", "userIntent": "..."}},
        {{"step": 3, "assistantMessage": "...", "userIntent": "..."}}
      ]
    }},
    {{
      "level": 2,
      "levelTitle": "...",
      "levelDescription": "...",
      "steps": [
        {{"step": 1, "assistantMessage": "...", "userIntent": "..."}},
        {{"step": 2, "assistantMessage": "...", "userIntent": "..."}},
        {{"step": 3, "assistantMessage": "...", "userIntent": "..."}}
      ]
    }},
    {{
      "level": 3,
      "levelTitle": "...",
      "levelDescription": "...",
      "steps": [
        {{"step": 1, "assistantMessage": "...", "userIntent": "..."}},
        {{"step": 2, "assistantMessage": "...", "userIntent": "..."}},
        {{"step": 3, "assistantMessage": "...", "userIntent": "..."}}
      ]
    }}
  ]
}}
"""

    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    text = response.output_text.strip()
    return json.loads(text)

#7. services/audio_service.py

In [ ]:
%%writefile services/audio_service.py
import requests
from pydub import AudioSegment


def download_audio_from_s3(s3_url: str, save_path: str):
    response = requests.get(s3_url, timeout=60)
    response.raise_for_status()

    with open(save_path, "wb") as f:
        f.write(response.content)

    return save_path


def preprocess_audio_to_mono_16k_wav(input_path: str, output_path: str):
    audio = AudioSegment.from_file(input_path)
    audio = audio.set_channels(1)
    audio = audio.set_frame_rate(16000)
    audio.export(output_path, format="wav")
    return output_path

#8. services/stt_service.py

In [ ]:
%%writefile services/stt_service.py
import whisper

model = None

def get_model():
    global model
    if model is None:
        model = whisper.load_model("large")
    return model

def transcribe_audio(audio_path: str) -> str:
    stt_model = get_model()
    result = stt_model.transcribe(audio_path, language="ko")
    return result["text"].strip()

#9. services/score_service.py

In [ ]:
%%writefile services/score_service.py
from difflib import SequenceMatcher
from jiwer import wer, cer
from openai import OpenAI
from config.settings import OPENAI_API_KEY
import json
from typing import List, Dict, Any, Optional

client = OpenAI(api_key=OPENAI_API_KEY)


# ----------------------------
# 기본 유틸
# ----------------------------
def split_korean_chars(text: str) -> List[str]:
    return [ch for ch in text.strip() if ch.strip()]


def safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def clamp(value, min_value, max_value):
    return max(min_value, min(max_value, value))


# ----------------------------
# 한글 자모 분해
# ----------------------------
CHOSUNG = [
    'ㄱ', 'ㄲ', 'ㄴ', 'ㄷ', 'ㄸ', 'ㄹ', 'ㅁ', 'ㅂ', 'ㅃ', 'ㅅ',
    'ㅆ', 'ㅇ', 'ㅈ', 'ㅉ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ'
]
JUNGSUNG = [
    'ㅏ', 'ㅐ', 'ㅑ', 'ㅒ', 'ㅓ', 'ㅔ', 'ㅕ', 'ㅖ', 'ㅗ', 'ㅘ',
    'ㅙ', 'ㅚ', 'ㅛ', 'ㅜ', 'ㅝ', 'ㅞ', 'ㅟ', 'ㅠ', 'ㅡ', 'ㅢ', 'ㅣ'
]
JONGSUNG = [
    '', 'ㄱ', 'ㄲ', 'ㄳ', 'ㄴ', 'ㄵ', 'ㄶ', 'ㄷ', 'ㄹ', 'ㄺ',
    'ㄻ', 'ㄼ', 'ㄽ', 'ㄾ', 'ㄿ', 'ㅀ', 'ㅁ', 'ㅂ', 'ㅄ', 'ㅅ',
    'ㅆ', 'ㅇ', 'ㅈ', 'ㅊ', 'ㅋ', 'ㅌ', 'ㅍ', 'ㅎ'
]


def decompose_hangul(char: str):
    if len(char) != 1 or not ('가' <= char <= '힣'):
        return None

    base = ord(char) - ord('가')
    cho = base // 588
    jung = (base % 588) // 28
    jong = base % 28

    return {
        "initial": CHOSUNG[cho],
        "medial": JUNGSUNG[jung],
        "final": JONGSUNG[jong]
    }


def analyze_syllable_difference(ref_char: str, hyp_char: str):
    ref_parts = decompose_hangul(ref_char)
    hyp_parts = decompose_hangul(hyp_char)

    if not ref_parts or not hyp_parts:
        return {
            "type": "unknown",
            "refParts": ref_parts,
            "hypParts": hyp_parts,
            "phonemeDiff": {
                "initial": "unknown",
                "medial": "unknown",
                "final": "unknown"
            }
        }

    diff_parts = []

    initial_state = "same" if ref_parts["initial"] == hyp_parts["initial"] else "different"
    medial_state = "same" if ref_parts["medial"] == hyp_parts["medial"] else "different"
    final_state = "same" if ref_parts["final"] == hyp_parts["final"] else "different"

    if initial_state == "different":
        diff_parts.append("initial")
    if medial_state == "different":
        diff_parts.append("medial")
    if final_state == "different":
        diff_parts.append("final")

    return {
        "type": "+".join(diff_parts) if diff_parts else "equal",
        "refParts": ref_parts,
        "hypParts": hyp_parts,
        "phonemeDiff": {
            "initial": initial_state,
            "medial": medial_state,
            "final": final_state
        }
    }


def build_articulation_hint(ref_char: str, hyp_char: str):
    diff = analyze_syllable_difference(ref_char, hyp_char)

    if diff["type"] == "medial":
        return {
            "issue": f"'{ref_char}' 부분의 모음이 다르게 인식되었을 수 있음",
            "suggestion": "입모양을 조금 더 분명하게 하여 천천히 발음해보세요."
        }

    if diff["type"] == "initial":
        initial = diff["refParts"]["initial"]
        if initial == "ㅁ":
            return {
                "issue": f"'{ref_char}' 부분의 초성 구분이 흐려졌을 수 있음",
                "suggestion": "입술을 충분히 붙인 뒤 소리를 내보세요."
            }
        if initial in ["ㅅ", "ㅈ", "ㅊ"]:
            return {
                "issue": f"'{ref_char}' 부분의 초성 구분이 흐려졌을 수 있음",
                "suggestion": "혀와 입모양을 조금 더 분명하게 하여 천천히 발음해보세요."
            }
        return {
            "issue": f"'{ref_char}' 부분의 초성 구분이 흐려졌을 수 있음",
            "suggestion": "첫소리를 조금 더 분명하게 내보세요."
        }

    if diff["type"] == "final":
        return {
            "issue": f"'{ref_char}' 부분의 받침이 약하게 들렸을 수 있음",
            "suggestion": "마지막 소리까지 힘을 유지해서 발음해보세요."
        }

    if diff["type"] == "initial+medial":
        return {
            "issue": f"'{ref_char}' 부분의 자음과 모음이 함께 다르게 인식되었을 수 있음",
            "suggestion": "입모양과 첫소리를 모두 조금 더 분명하게 하여 천천히 발음해보세요."
        }

    if diff["type"] == "initial+final":
        return {
            "issue": f"'{ref_char}' 부분의 첫소리와 받침이 흐려졌을 수 있음",
            "suggestion": "첫소리와 마지막 소리를 모두 분명하게 발음해보세요."
        }

    if diff["type"] == "medial+final":
        return {
            "issue": f"'{ref_char}' 부분의 모음과 받침이 함께 흐려졌을 수 있음",
            "suggestion": "입모양을 분명하게 하고 마지막 소리까지 유지해보세요."
        }

    if diff["type"] == "initial+medial+final":
        return {
            "issue": f"'{ref_char}' 부분 전체가 다르게 인식되었을 수 있음",
            "suggestion": "해당 음절을 천천히 끊어서 또렷하게 연습해보세요."
        }

    return {
        "issue": f"'{ref_char}' 부분이 '{hyp_char}'처럼 다르게 인식되었음",
        "suggestion": "천천히 또렷하게 발음해보세요."
    }


# ----------------------------
# 점수 계산
# ----------------------------
def calculate_reference_scores(reference_text: str, stt_text: str):
    reference_text = reference_text.strip()
    stt_text = stt_text.strip()

    if not reference_text:
        return {
            "pronunciationScore": 0.0,
            "similarityScore": 0.0,
            "wer": 1.0,
            "cer": 1.0
        }

    wer_score = wer(reference_text, stt_text)
    cer_score = cer(reference_text, stt_text)
    similarity = SequenceMatcher(None, reference_text, stt_text).ratio()

    pronunciation_score = round(max(0, 100 * (1 - wer_score)), 2)
    similarity_score = round(similarity * 100, 2)

    return {
        "pronunciationScore": pronunciation_score,
        "similarityScore": similarity_score,
        "wer": round(wer_score, 4),
        "cer": round(cer_score, 4)
    }


# ----------------------------
# 문자열 정렬 분석
# ----------------------------
def build_alignment(reference_text: str, stt_text: str) -> Dict[str, Any]:
    reference_chars = split_korean_chars(reference_text)
    stt_chars = split_korean_chars(stt_text)

    matcher = SequenceMatcher(None, reference_chars, stt_chars)
    opcodes = matcher.get_opcodes()

    aligned = []
    mismatch_indexes = []
    ref_index = 0

    for tag, i1, i2, j1, j2 in opcodes:
        if tag == "equal":
            for k in range(i2 - i1):
                aligned.append({
                    "refIndex": ref_index,
                    "ref": reference_chars[i1 + k],
                    "hyp": stt_chars[j1 + k],
                    "type": "equal"
                })
                ref_index += 1

        elif tag == "replace":
            ref_chunk = reference_chars[i1:i2]
            hyp_chunk = stt_chars[j1:j2]
            max_len = max(len(ref_chunk), len(hyp_chunk))

            for k in range(max_len):
                ref_char = ref_chunk[k] if k < len(ref_chunk) else ""
                hyp_char = hyp_chunk[k] if k < len(hyp_chunk) else ""

                if ref_char and hyp_char:
                    aligned.append({
                        "refIndex": ref_index,
                        "ref": ref_char,
                        "hyp": hyp_char,
                        "type": "substitute"
                    })
                    mismatch_indexes.append(ref_index)
                    ref_index += 1
                elif ref_char and not hyp_char:
                    aligned.append({
                        "refIndex": ref_index,
                        "ref": ref_char,
                        "hyp": "",
                        "type": "delete"
                    })
                    mismatch_indexes.append(ref_index)
                    ref_index += 1
                elif not ref_char and hyp_char:
                    aligned.append({
                        "refIndex": max(ref_index - 1, 0),
                        "ref": "",
                        "hyp": hyp_char,
                        "type": "insert"
                    })

        elif tag == "delete":
            for k in range(i1, i2):
                aligned.append({
                    "refIndex": ref_index,
                    "ref": reference_chars[k],
                    "hyp": "",
                    "type": "delete"
                })
                mismatch_indexes.append(ref_index)
                ref_index += 1

        elif tag == "insert":
            for k in range(j1, j2):
                aligned.append({
                    "refIndex": max(ref_index - 1, 0),
                    "ref": "",
                    "hyp": stt_chars[k],
                    "type": "insert"
                })

    return {
        "referenceChars": reference_chars,
        "sttChars": stt_chars,
        "aligned": aligned,
        "mismatchIndexes": mismatch_indexes
    }


# ----------------------------
# 규칙 기반 피드백 생성
# ----------------------------
def build_rule_based_analysis(reference_text: str, alignment_result):
    ref_chars = split_korean_chars(reference_text)

    word_analysis = []
    inserts = []
    insert_feedbacks = []

    ref_index = 0

    for pair in alignment_result:
        ref_char = pair.get("ref", "")
        hyp_char = pair.get("hyp", "")
        status = pair.get("status", "")

        if status == "insert":
            inserts.append(hyp_char)
            insert_feedbacks.append(f"'{hyp_char}' 발화가 추가로 들어갔을 수 있음")
            continue

        item = {
            "text": ref_char,
            "refChar": ref_char,
            "hypChar": hyp_char,
            "status": status,
            "score": 0,
            "issue": "",
            "suggestion": "",
            "errorType": None,
            "refParts": decompose_hangul(ref_char) if ref_char else None,
            "hypParts": decompose_hangul(hyp_char) if hyp_char else None,
            "phonemeDiff": {
                "initial": "same" if ref_char == hyp_char and ref_char else "unknown",
                "medial": "same" if ref_char == hyp_char and ref_char else "unknown",
                "final": "same" if ref_char == hyp_char and ref_char else "unknown"
            }
        }

        if status == "equal":
            item["score"] = 100
            item["issue"] = "정확하게 발음되었음"
            item["suggestion"] = "좋습니다. 지금처럼 유지해보세요."
            if item["refParts"] and item["hypParts"]:
                item["phonemeDiff"] = {
                    "initial": "same",
                    "medial": "same",
                    "final": "same"
                }

        elif status == "replace":
            diff = analyze_syllable_difference(ref_char, hyp_char)
            hint = build_articulation_hint(ref_char, hyp_char)

            item["score"] = 40
            item["issue"] = hint["issue"]
            item["suggestion"] = hint["suggestion"]
            item["errorType"] = diff["type"]
            item["refParts"] = diff.get("refParts")
            item["hypParts"] = diff.get("hypParts")
            item["phonemeDiff"] = diff.get("phonemeDiff", {
                "initial": "unknown",
                "medial": "unknown",
                "final": "unknown"
            })

            if diff["type"] == "equal":
                item["score"] = 100
                item["issue"] = "정확하게 발음되었음"
                item["suggestion"] = "좋습니다. 지금처럼 유지해보세요."

        elif status == "delete":
            item["score"] = 0
            item["issue"] = f"'{ref_char}' 발음이 누락되었을 수 있음"
            item["suggestion"] = "해당 글자를 한 번 더 또렷하게 발음해보세요."
            item["errorType"] = "delete"
            item["hypParts"] = None
            item["phonemeDiff"] = {
                "initial": "missing",
                "medial": "missing",
                "final": "missing"
            }

        else:
            item["score"] = 0
            item["issue"] = "분석 불가"
            item["suggestion"] = "다시 한 번 천천히 발음해보세요."
            item["errorType"] = "unknown"

        word_analysis.append(item)
        ref_index += 1

    return word_analysis, inserts, insert_feedbacks

def build_overall_rule_feedback(word_analysis: List[Dict[str, Any]], insert_feedbacks: List[str]) -> str:
    deletes = sum(1 for x in word_analysis if x["errorType"] == "delete")
    substitutes = sum(1 for x in word_analysis if x["errorType"] == "substitute")
    tail_weak = sum(
        1 for x in word_analysis[-2:]
        if x["errorType"] in ["delete", "substitute"]
    ) if len(word_analysis) >= 2 else 0

    messages = []

    if deletes > 0:
        messages.append("일부 음절이 생략되거나 약하게 발음된 부분이 있었음")
    if substitutes > 0:
        messages.append("일부 음절이 다른 소리로 인식된 부분이 있었음")
    if tail_weak > 0:
        messages.append("문장 끝부분이 약해졌을 가능성이 있음")
    if insert_feedbacks:
        messages.append("발화가 조금 늘어지거나 불필요한 소리가 섞였을 수 있음")

    if not messages:
        return "전반적으로 목표 문장과 유사하게 발화되었음."

    return ". ".join(messages) + "."


# ----------------------------
# WhisperX timestamp 매핑
# ----------------------------
def attach_syllable_timestamps(
    word_analysis: List[Dict[str, Any]],
    reference_text: str,
    whisperx_words: Optional[List[Dict[str, Any]]] = None
):
    if not whisperx_words:
        for item in word_analysis:
            item["start"] = None
            item["end"] = None
        return word_analysis

    ref_chars = split_korean_chars(reference_text)
    char_pointer = 0

    for item in word_analysis:
        item["start"] = None
        item["end"] = None

    for word_info in whisperx_words:
        word = str(word_info.get("word", "")).replace(" ", "")
        start = word_info.get("start")
        end = word_info.get("end")

        if not word or start is None or end is None:
            continue

        syllables = split_korean_chars(word)
        if not syllables:
            continue

        duration = max(0.0, float(end) - float(start))
        unit = duration / len(syllables) if len(syllables) > 0 else 0.0

        for i, _ in enumerate(syllables):
            if char_pointer >= len(ref_chars) or char_pointer >= len(word_analysis):
                break

            word_analysis[char_pointer]["start"] = round(float(start) + unit * i, 3)
            word_analysis[char_pointer]["end"] = round(float(start) + unit * (i + 1), 3)
            char_pointer += 1

    return word_analysis


# ----------------------------
# LLM 평가 (시나리오 전용)
# ----------------------------
def evaluate_with_llm(step_content: str, stt_text: str):
    system_prompt = """
너는 성인 언어 재활 보조 평가자임.

사용자의 발화(STT 결과)를 보고 아래만 평가해야 함.
1. 사용자가 말하려던 의도 문장 추정
2. step 내용과 비교했을 때 의미 전달률(0~100)
3. 전체 발화에 대한 짧은 피드백

반드시 JSON만 반환해야 함.

반환 형식:
{
  "inferredReferenceText": "...",
  "meaningDeliveryScore": 0,
  "feedback": "..."
}
"""

    user_prompt = f"""
[원래 연습 목표 문장/상황]
{step_content}

[사용자 STT 결과]
{stt_text}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return json.loads(response.choices[0].message.content)


# ----------------------------
# reference 전용 평가
# ----------------------------
def evaluate_reference_response(
    reference_text: str,
    stt_text: str,
    whisperx_words: Optional[List[Dict[str, Any]]] = None
):
    reference_text = reference_text.strip()
    stt_text = stt_text.strip()

    reference_scores = calculate_reference_scores(reference_text, stt_text)
    alignment_result = build_alignment(reference_text, stt_text)

    word_analysis, inserts, insert_feedbacks = build_rule_based_analysis(
        reference_text,
        alignment_result
    )

    word_analysis = attach_syllable_timestamps(
        word_analysis=word_analysis,
        reference_text=reference_text,
        whisperx_words=whisperx_words
    )

    rule_feedback = build_overall_rule_feedback(word_analysis, insert_feedbacks)

    avg_word_score = round(
        sum(item["score"] for item in word_analysis) / len(word_analysis),
        2
    ) if word_analysis else 0.0

    return {
        "referenceText": reference_text,
        "sttText": stt_text,
        "referenceSource": "user_input",
        "evaluationMode": "reference_only",
        "meaningDeliveryScore": round(float(reference_scores.get("similarityScore", 0.0)), 2),
        "pronunciationScore": round(
            (
                float(reference_scores.get("pronunciationScore", 0)) +
                float(avg_word_score)
            ) / 2,
            2
        ),
        "feedback": rule_feedback,
        "wordAnalysis": word_analysis,
        "diffAnalysis": alignment_result,
        "insertions": inserts,
        "similarityScore": reference_scores.get("similarityScore", 0.0),
        "wer": reference_scores.get("wer", 1.0),
        "cer": reference_scores.get("cer", 1.0)
    }


# ----------------------------
# 시나리오 전용 평가
# ----------------------------
def evaluate_scenario_response(
    step_content: str,
    stt_text: str,
    whisperx_words: Optional[List[Dict[str, Any]]] = None
):
    llm_result = evaluate_with_llm(step_content, stt_text)

    reference_text = llm_result.get("inferredReferenceText", "").strip()
    meaning_delivery_score = clamp(
        safe_int(llm_result.get("meaningDeliveryScore", 0), 0),
        0,
        100
    )
    llm_feedback = llm_result.get("feedback", "").strip()

    reference_scores = calculate_reference_scores(reference_text, stt_text)
    alignment_result = build_alignment(reference_text, stt_text)

    word_analysis, inserts, insert_feedbacks = build_rule_based_analysis(
        reference_text,
        alignment_result
    )

    word_analysis = attach_syllable_timestamps(
        word_analysis=word_analysis,
        reference_text=reference_text,
        whisperx_words=whisperx_words
    )

    rule_feedback = build_overall_rule_feedback(word_analysis, insert_feedbacks)

    avg_word_score = round(
        sum(item["score"] for item in word_analysis) / len(word_analysis),
        2
    ) if word_analysis else 0.0

    final_feedback = f"{llm_feedback} {rule_feedback}".strip() if llm_feedback else rule_feedback

    return {
        "referenceText": reference_text,
        "sttText": stt_text,
        "referenceSource": "llm",
        "evaluationMode": "scenario_llm_only",
        "meaningDeliveryScore": meaning_delivery_score,
        "pronunciationScore": round(
            (
                float(reference_scores.get("pronunciationScore", 0)) +
                float(avg_word_score)
            ) / 2,
            2
        ),
        "feedback": final_feedback,
        "wordAnalysis": word_analysis,
        "diffAnalysis": alignment_result,
        "insertions": inserts,
        "similarityScore": reference_scores.get("similarityScore", 0.0),
        "wer": reference_scores.get("wer", 1.0),
        "cer": reference_scores.get("cer", 1.0)
    }

#10. services/chat_service

In [ ]:
%%writefile services/chat_service.py
from openai import OpenAI
from config.settings import OPENAI_API_KEY
from typing import List, Dict, Optional

client = OpenAI(api_key=OPENAI_API_KEY)


def build_context_messages(
    chat_history: Optional[List[Dict[str, str]]] = None,
    max_pairs: int = 3
) -> List[Dict[str, str]]:
    """
    chat_history 예시:
    [
        {"role": "user", "content": "안녕하세요"},
        {"role": "assistant", "content": "안녕하세요. 오늘은 어떤 연습을 해볼까요?"},
        {"role": "user", "content": "병원 예약 연습이 어려워요"},
        {"role": "assistant", "content": "괜찮아요. 천천히 같이 해봐요."}
    ]

    규칙:
    - role은 user / assistant만 허용
    - assistant에는 '답변'만 저장되어 있어야 함 (피드백 제외)
    - 최근 max_pairs쌍만 사용
    - 한 쌍 = user 1개 + assistant 1개
    - max_pairs=3 이면 총 6개 메시지까지 들어감
    """
    if not chat_history:
        return []

    allowed_roles = {"user", "assistant"}

    normalized = []
    for item in chat_history:
        role = item.get("role", "").strip()
        content = item.get("content", "").strip()

        if role not in allowed_roles:
            continue
        if not content:
            continue

        normalized.append({
            "role": role,
            "content": content
        })

    # 최근 user-assistant 쌍 최대 max_pairs개만 유지
    pairs = []
    current_user = None

    for msg in normalized:
        if msg["role"] == "user":
            current_user = msg
        elif msg["role"] == "assistant" and current_user is not None:
            pairs.append([current_user, msg])
            current_user = None

    recent_pairs = pairs[-max_pairs:]

    context_messages = []
    for user_msg, assistant_msg in recent_pairs:
        context_messages.append(user_msg)
        context_messages.append(assistant_msg)

    return context_messages


def parse_reply_and_feedback(raw_text: str) -> Dict[str, str]:
    """
    모델이 아래 형식으로 응답했다고 가정:
    답변: ...
    피드백: ...

    형식이 조금 어긋나도 최대한 안전하게 파싱함.
    """
    raw_text = raw_text.strip()

    reply_prefix = "답변:"
    feedback_prefix = "피드백:"

    reply = ""
    feedback = ""

    if reply_prefix in raw_text and feedback_prefix in raw_text:
        reply_start = raw_text.find(reply_prefix) + len(reply_prefix)
        feedback_start = raw_text.find(feedback_prefix)

        reply = raw_text[reply_start:feedback_start].strip()
        feedback = raw_text[feedback_start + len(feedback_prefix):].strip()
    else:
        # 형식이 깨졌을 때 fallback
        lines = [line.strip() for line in raw_text.splitlines() if line.strip()]
        if len(lines) >= 2:
            reply = lines[0]
            feedback = " ".join(lines[1:])
        elif len(lines) == 1:
            reply = lines[0]
            feedback = "천천히 말해도 괜찮아요."
        else:
            reply = "천천히 다시 말해주셔도 괜찮아요."
            feedback = "좋아요. 부담 없이 이어가면 돼요."

    return {
        "reply": reply,
        "feedback": feedback
    }


def generate_free_talk_reply(
    user_message: str,
    chat_history: Optional[List[Dict[str, str]]] = None
) -> Dict[str, str]:
    """
    반환 예시:
    {
        "reply": "괜찮아요. 병원 예약은 차근차근 연습하면 돼요.",
        "feedback": "문장을 끝까지 말하려고 한 점이 좋았어요.",
        "assistant_message_for_history": "괜찮아요. 병원 예약은 차근차근 연습하면 돼요."
    }

    주의:
    - chat_history에는 사용자 메시지와 assistant의 '답변(reply)'만 저장해야 함
    - feedback은 화면 표시용이고, 다음 LLM 호출의 문맥에는 넣지 않음
    """
    system_prompt = """
너는 성인 언어장애/조음장애 사용자를 위한 따뜻한 대화 파트너임.

목표:
- 사용자가 편안하게 말하도록 돕기
- 짧고 쉬운 문장으로 답하기
- 실수보다 자신감을 살리는 방향으로 돕기
- 이전 대화 맥락이 있으면 자연스럽게 이어가기

반드시 아래 형식으로만 출력:
답변: <사용자에게 보여줄 따뜻한 답변 1~3문장>
피드백: <아주 짧은 긍정 피드백 1문장>

추가 규칙:
1. 답변은 너무 길지 않게 작성
2. 쉬운 문장 사용
3. 부담 주지 않기
4. 필요하면 천천히 다시 말해보라고 부드럽게 유도
5. 피드백은 짧고 긍정적으로 작성
6. 피드백에는 교정보다 잘한 점 중심으로 작성
7. 한 번에 질문을 너무 많이 하지 않기
"""

    messages = [{"role": "system", "content": system_prompt}]

    # 최근 3쌍(총 6개 메시지)만 문맥으로 사용
    messages.extend(build_context_messages(chat_history, max_pairs=3))

    # 현재 사용자 메시지 추가
    messages.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.8,
        messages=messages
    )

    raw_content = response.choices[0].message.content.strip()
    parsed = parse_reply_and_feedback(raw_content)

    return {
        "reply": parsed["reply"],
        "feedback": parsed["feedback"],
        # 히스토리에는 피드백 제외한 답변만 저장
        "assistant_message_for_history": parsed["reply"]
    }

#11. api/app.py

In [ ]:
%%writefile api/app.py
import os
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

from config.settings import COLAB_API_TOKEN
from services.scenario_service import generate_scenario_levels
from services.audio_service import download_audio_from_s3, preprocess_audio_to_mono_16k_wav
from services.stt_service import transcribe_audio
from services.score_service import evaluate_reference_response, evaluate_scenario_response
from services.chat_service import generate_free_talk_reply

app = FastAPI()

INPUT_DIR = "data/input"
OUTPUT_DIR = "data/output"

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


class ScenarioRequest(BaseModel):
    scenarioContext: str
    goal: str


class ReferencePracticeRequest(BaseModel):
    s3Url: str
    referenceText: str


class FreeTalkRequest(BaseModel):
    userMessage: str
    chatHistory: list[dict] | None = None


class ScenarioPracticeRequest(BaseModel):
    s3Url: str
    level: int
    step: int
    stepContent: str


def validate_token(x_api_token: str | None):
    if COLAB_API_TOKEN and x_api_token != COLAB_API_TOKEN:
        raise HTTPException(status_code=401, detail="Unauthorized")


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/generate-scenario")
def generate_scenario(req: ScenarioRequest, x_api_token: str | None = Header(default=None)):
    validate_token(x_api_token)

    try:
        result = generate_scenario_levels(req.scenarioContext, req.goal)
        return {
            "success": True,
            "data": result
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/practice/reference")
def practice_reference(req: ReferencePracticeRequest, x_api_token: str | None = Header(default=None)):
    validate_token(x_api_token)

    raw_path = os.path.join(INPUT_DIR, "reference_input_audio")
    wav_path = os.path.join(OUTPUT_DIR, "reference_input.wav")

    try:
        download_audio_from_s3(req.s3Url, raw_path)
        preprocess_audio_to_mono_16k_wav(raw_path, wav_path)
        stt_text = transcribe_audio(wav_path)

        eval_result = evaluate_reference_response(
            reference_text=req.referenceText,
            stt_text=stt_text
        )

        return {
            "success": True,
            "mode": "reference_practice",
            **eval_result
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/chat/free-talk")
def free_talk(req: FreeTalkRequest, x_api_token: str | None = Header(default=None)):
    """
    AI 자유대화
    사용자 메시지에 대해
    1. 답변
    2. 짧은 피드백
    을 반환함
    """
    validate_token(x_api_token)

    try:
        result = generate_free_talk_reply(
            user_message=req.userMessage,
            chat_history=req.chatHistory
        )

        return {
            "success": True,
            "mode": "free_talk",
            "userMessage": req.userMessage,
            "aiReply": result["reply"],
            "aiFeedback": result["feedback"],
            "assistantMessageForHistory": result["assistant_message_for_history"]
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.post("/practice/scenario")
def practice_scenario(req: ScenarioPracticeRequest, x_api_token: str | None = Header(default=None)):
    """
    시나리오 연습
    referenceText 없이 step 내용을 바탕으로
    LLM이 의도 문장을 추정하고
    발음 정확도 + 의미 전달률 반환
    """
    validate_token(x_api_token)

    raw_path = os.path.join(INPUT_DIR, "scenario_input_audio")
    wav_path = os.path.join(OUTPUT_DIR, "scenario_input.wav")

    try:
        download_audio_from_s3(req.s3Url, raw_path)
        preprocess_audio_to_mono_16k_wav(raw_path, wav_path)
        stt_text = transcribe_audio(wav_path)
        eval_result = evaluate_scenario_response(req.stepContent, stt_text)

        return {
            "success": True,
            "mode": "scenario_practice",
            "level": req.level,
            "step": req.step,
            "stepContent": req.stepContent,
            "sttText": stt_text,
            **eval_result
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))